<a href="https://colab.research.google.com/github/siddharth-0309/gold-price-forecast/blob/main/gold_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install yfinance**

In [ ]:
pip install yfinance

**Load Data**

In [ ]:
import pandas as pd
import yfinance as yf

tickers = {
    'gold': 'GC=F',       # actual gold futures, not GLD
    'dxy': 'DX-Y.NYB',
    'oil': 'CL=F',
    'yield10y': '^TNX',
    'sp500': '^GSPC'
}

data_dict = {}
for name, ticker in tickers.items():
    df = yf.Ticker(ticker).history(period='5y')['Close']
    df.index = df.index.date
    df.index = pd.to_datetime(df.index)
    data_dict[name] = df

combined = pd.DataFrame(data_dict)
combined = combined.sort_index()

print(combined.tail(10).to_markdown())
print(combined.isna().sum())
print(f"\nTotal rows: {len(combined)}")

combined.to_csv('market_data.csv')

|                     |   gold |    dxy |   oil |   yield10y |   sp500 |
|:--------------------|-------:|-------:|------:|-----------:|--------:|
| 2026-07-06 00:00:00 | 4155.1 | 100.85 | 68.55 |      4.479 | 7537.43 |
| 2026-07-07 00:00:00 | 4145.3 | 101.14 | 70.44 |      4.529 | 7503.85 |
| 2026-07-08 00:00:00 | 4070.9 | 101.05 | 73.52 |      4.569 | 7482.71 |
| 2026-07-09 00:00:00 | 4130.6 | 100.94 | 72.08 |      4.539 | 7543.64 |
| 2026-07-10 00:00:00 | 4104.1 | 100.97 | 71.41 |      4.569 | 7575.39 |
| 2026-07-13 00:00:00 | 3997   | 101.28 | 78.14 |      4.609 | 7515.34 |
| 2026-07-14 00:00:00 | 4061.1 | 100.94 | 79.34 |      4.585 | 7543.59 |
| 2026-07-15 00:00:00 | 4044   | 100.5  | 79.6  |      4.545 | 7572.4  |
| 2026-07-16 00:00:00 | 3985.6 | 100.73 | 78.95 |      4.569 | 7533.77 |
| 2026-07-17 00:00:00 | 4018.8 | 100.75 | 81.78 |      4.541 | 7457.69 |
gold        1
dxy         1
oil         1
yield10y    3
sp500       3
dtype: int64

Total rows: 1258


In [ ]:
combined_clean = combined.ffill()
combined_clean = combined_clean.dropna()

print(f"Before cleaning: {len(combined)} rows")
print(f"After cleaning: {len(combined_clean)} rows")
print(combined_clean.isna().sum())

combined_clean.to_csv('market_data_clean.csv')

Before cleaning: 1258 rows
After cleaning: 1258 rows
gold        0
dxy         0
oil         0
yield10y    0
sp500       0
dtype: int64


In [ ]:
df = combined_clean.copy()

# 1. Lag features - gold ke pichle din ke prices
for lag in [1, 3, 5, 7]:
    df[f'gold_lag_{lag}'] = df['gold'].shift(lag)

# 2. Rolling stats - trend aur volatility capture karne ke liye
df['gold_roll_mean_7'] = df['gold'].rolling(7).mean()
df['gold_roll_std_7'] = df['gold'].rolling(7).std()

# 3. Related asset lags (1 din pehle ka value - taaki future leakage na ho)
for col in ['dxy', 'oil', 'yield10y', 'sp500']:
    df[f'{col}_lag_1'] = df[col].shift(1)

# 4. Day of week (Monday effect jaisa kuch capture karne ke liye)
df['day_of_week'] = df.index.dayofweek

# 5. TARGETS - agle 1, 2, 3 din ka gold price (ye predict karna hai)
df['target_day1'] = df['gold'].shift(-1)
df['target_day2'] = df['gold'].shift(-2)
df['target_day3'] = df['gold'].shift(-3)

# Lag/rolling ki wajah se shuru ke rows mein NaN aayenge, aur target ki wajah se aakhri rows mein
df_features = df.dropna()

print(f"Rows before feature engineering: {len(df)}")
print(f"Rows after dropping NaN (from lags/targets): {len(df_features)}")
print(df_features.tail(10).to_markdown())

df_features.to_csv('gold_features.csv')

Rows before feature engineering: 1258
Rows after dropping NaN (from lags/targets): 1248
|                     |   gold |    dxy |   oil |   yield10y |   sp500 |   gold_lag_1 |   gold_lag_3 |   gold_lag_5 |   gold_lag_7 |   gold_roll_mean_7 |   gold_roll_std_7 |   dxy_lag_1 |   oil_lag_1 |   yield10y_lag_1 |   sp500_lag_1 |   day_of_week |   target_day1 |   target_day2 |   target_day3 |
|:--------------------|-------:|-------:|------:|-----------:|--------:|-------------:|-------------:|-------------:|-------------:|-------------------:|------------------:|------------:|------------:|-----------------:|--------------:|--------------:|--------------:|--------------:|--------------:|
| 2026-06-30 00:00:00 | 4022.9 | 101.19 | 69.5  |      4.418 | 7499.36 |       4022.3 |       4030.5 |       4129.9 |       4224.1 |            4065.21 |           68.8367 |      101.11 |       70.75 |            4.374 |       7440.43 |             1 |        4068.3 |        4112.7 |        4155.1 |
| 2026-07

In [ ]:
# Time-based split - RANDOM split nahi karna (time series leakage se bachna hai)
split_ratio = 0.85  # 85% train, 15% test (recent data test ke liye)
split_idx = int(len(df_features) * split_ratio)

train = df_features.iloc[:split_idx]
test = df_features.iloc[split_idx:]

print(f"Train: {len(train)} rows ({train.index.min()} to {train.index.max()})")
print(f"Test: {len(test)} rows ({test.index.min()} to {test.index.max()})")

# Features aur targets alag karo
feature_cols = [col for col in df_features.columns if col not in ['target_day1', 'target_day2', 'target_day3']]

X_train = train[feature_cols]
X_test = test[feature_cols]

y_train_day1 = train['target_day1']
y_train_day2 = train['target_day2']
y_train_day3 = train['target_day3']

y_test_day1 = test['target_day1']
y_test_day2 = test['target_day2']
y_test_day3 = test['target_day3']

print(f"\nFeature columns ({len(feature_cols)}): {feature_cols}")

Train: 1060 rows (2021-07-28 00:00:00 to 2025-10-10 00:00:00)
Test: 188 rows (2025-10-13 00:00:00 to 2026-07-14 00:00:00)

Feature columns (16): ['gold', 'dxy', 'oil', 'yield10y', 'sp500', 'gold_lag_1', 'gold_lag_3', 'gold_lag_5', 'gold_lag_7', 'gold_roll_mean_7', 'gold_roll_std_7', 'dxy_lag_1', 'oil_lag_1', 'yield10y_lag_1', 'sp500_lag_1', 'day_of_week']


In [ ]:
pip install lightgbm

In [ ]:
import lightgbm as lgb

quantiles = [0.1, 0.5, 0.9]  # low, median, high
targets = {
    'day1': (y_train_day1, y_test_day1),
    'day2': (y_train_day2, y_test_day2),
    'day3': (y_train_day3, y_test_day3)
}

models = {}  # sab trained models yahan store honge

for horizon, (y_train, y_test) in targets.items():
    for q in quantiles:
        model = lgb.LGBMRegressor(
            objective='quantile',
            alpha=q,              # quantile regression ka main parameter
            n_estimators=150,     # Reverted to 150 for higher coverage
            learning_rate=0.03,
            max_depth=3,
            min_child_samples=30,     # Reverted to 30 for higher coverage
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose=-1
        )
        model.fit(X_train, y_train)
        models[f'{horizon}_q{int(q*100)}'] = model
        print(f"Trained: {horizon} - quantile {q}")

print(f"\nTotal models trained: {len(models)}")

Trained: day1 - quantile 0.1
Trained: day1 - quantile 0.5
Trained: day1 - quantile 0.9
Trained: day2 - quantile 0.1
Trained: day2 - quantile 0.5
Trained: day2 - quantile 0.9
Trained: day3 - quantile 0.1
Trained: day3 - quantile 0.5
Trained: day3 - quantile 0.9

Total models trained: 9


### Why use LightGBM and Quantile Regression?

**Why was LightGBM the best fit here?**

*   **Small-to-medium datasets:** It performs well on small to medium-sized datasets (around 1000 rows).
*   **Non-linear interactions:** It automatically captures non-linear interactions between multiple features, reducing the need for manual feature engineering.
*   **Fast training and easy tuning:** LightGBM trains models quickly and its parameters are easy to tune.
*   **Built-in support for Quantile Regression:** It supports Quantile Regression with the 'objective=\'quantile\'' parameter, which was our core requirement (i.e., range prediction).

**Why was Quantile Regression used?**

*   **Direct percentile training:** It directly trains for each percentile.
*   **Adaptive:** It naturally widens or narrows the prediction range based on market conditions.
*   **Seamless integration with LightGBM:** It works seamlessly with LightGBM.

### Why use LightGBM and Quantile Regression?

**Why was LightGBM the best fit here?**

*   **Small-to-medium datasets:** It performs well on small to medium-sized datasets (around 1000 rows).
*   **Non-linear interactions:** It automatically captures non-linear interactions between multiple features, reducing the need for manual feature engineering.
*   **Fast training and easy tuning:** LightGBM trains models quickly and its parameters are easy to tune.
*   **Built-in support for Quantile Regression:** It supports Quantile Regression with the 'objective=\'quantile\'' parameter, which was our core requirement (i.e., range prediction).

**Why was Quantile Regression used?**

*   **Direct percentile training:** It directly trains for each percentile.
*   **Adaptive:** It naturally widens or narrows the prediction range based on market conditions.
*   **Seamless integration with LightGBM:** It works seamlessly with LightGBM.

In [ ]:
import numpy as np

def pinball_loss(y_true, y_pred, quantile):
    diff = y_true - y_pred
    return np.mean(np.maximum(quantile * diff, (quantile - 1) * diff))

results = []

for horizon, (y_train, y_test) in targets.items():
    preds = {}
    for q in quantiles:
        model = models[f'{horizon}_q{int(q*100)}']
        preds[q] = model.predict(X_test)

    # Pinball loss for each quantile
    for q in quantiles:
        loss = pinball_loss(y_test.values, preds[q], q)
        results.append({'horizon': horizon, 'quantile': q, 'pinball_loss': round(loss, 3)})

    # Coverage check - actual value predicted range (10th-90th) ke andar aaya kitni baar
    within_range = np.mean((y_test.values >= preds[0.1]) & (y_test.values <= preds[0.9]))
    print(f"{horizon}: {within_range*100:.1f}% of actual values fell within predicted 10-90 range")

    # Median prediction ka average error
    mae = np.mean(np.abs(y_test.values - preds[0.5]))
    print(f"  Median prediction avg error: ${mae:.2f}\n")

results_df = pd.DataFrame(results)
print(results_df.to_markdown())

day1: 2.1% of actual values fell within predicted 10-90 range
  Median prediction avg error: $761.09

day2: 4.8% of actual values fell within predicted 10-90 range
  Median prediction avg error: $704.68

day3: 9.0% of actual values fell within predicted 10-90 range
  Median prediction avg error: $780.68

|    | horizon   |   quantile |   pinball_loss |
|---:|:----------|-----------:|---------------:|
|  0 | day1      |        0.1 |         85.199 |
|  1 | day1      |        0.5 |        380.544 |
|  2 | day1      |        0.9 |        507.029 |
|  3 | day2      |        0.1 |         82.336 |
|  4 | day2      |        0.5 |        352.338 |
|  5 | day2      |        0.9 |        498.585 |
|  6 | day3      |        0.1 |         82.741 |
|  7 | day3      |        0.5 |        390.339 |
|  8 | day3      |        0.9 |        445.594 |


------------------------------

In [ ]:
df = combined_clean.copy()

# 1. Lag features - % change based (price level pe depend nahi karte)
for lag in [1, 3, 5, 7]:
    df[f'gold_pct_lag_{lag}'] = df['gold'].pct_change(lag)

# 2. Rolling stats - % change ki volatility/trend
df['gold_pct_change_1d'] = df['gold'].pct_change(1)
df['gold_roll_mean_7'] = df['gold_pct_change_1d'].rolling(7).mean()
df['gold_roll_std_7'] = df['gold_pct_change_1d'].rolling(7).std()

# 3. Related assets - inko bhi % change mein convert karo, lagged
for col in ['dxy', 'oil', 'yield10y', 'sp500']:
    df[f'{col}_pct_lag_1'] = df[col].pct_change(1).shift(1)

# 4. Day of week
df['day_of_week'] = df.index.dayofweek

# 5. TARGETS - ab % change hai, absolute price nahi
df['target_day1'] = df['gold'].shift(-1) / df['gold'] - 1
df['target_day2'] = df['gold'].shift(-2) / df['gold'] - 1
df['target_day3'] = df['gold'].shift(-3) / df['gold'] - 1

# current gold price bhi rakho - baad mein price wapis banane ke liye
df['current_gold_price'] = df['gold']

df_features = df.dropna()
print(f"Rows after feature engineering: {len(df_features)}")
df_features.to_csv('gold_features_v2.csv')

Rows after feature engineering: 1248


### % Change vs Log Returns

Why did we use 'simple % change' here instead of 'log returns'? There are a few reasons:

*   **Simplicity:** At this initial stage, 'simple % change' is easier to understand and explain. It's good enough for creating a working model.
*   **Improved results:** Using 'simple % change' has already significantly improved the numbers.

**Future Improvement:** If further precision is needed, 'log returns' could be a valid next step. You can mention this as a 'future improvement' in your case study.

### % Change vs Log Returns

Why did we use 'simple % change' here instead of 'log returns'? There are a few reasons:

*   **Simplicity:** At this initial stage, 'simple % change' is easier to understand and explain. It's good enough for creating a working model.
*   **Improved results:** Using 'simple % change' has already significantly improved the numbers.

**Future Improvement:** If further precision is needed, 'log returns' could be a valid next step. You can mention this as a 'future improvement' in your case study.

In [ ]:
split_idx = int(len(df_features) * 0.85)
train = df_features.iloc[:split_idx]
test = df_features.iloc[split_idx:]

feature_cols = [c for c in df_features.columns if c not in ['target_day1','target_day2','target_day3','current_gold_price']]

X_train = train[feature_cols]
X_test = test[feature_cols]

y_train_day1, y_test_day1 = train['target_day1'], test['target_day1']
y_train_day2, y_test_day2 = train['target_day2'], test['target_day2']
y_train_day3, y_test_day3 = train['target_day3'], test['target_day3']

print(f"Train: {len(train)}, Test: {len(test)}")

Train: 1060, Test: 188


In [ ]:
targets = {
    'day1': (y_train_day1, y_test_day1),
    'day2': (y_train_day2, y_test_day2),
    'day3': (y_train_day3, y_test_day3)
}

models = {}
for horizon, (y_train, y_test) in targets.items():
    for q in [0.1, 0.5, 0.9]:
        model = lgb.LGBMRegressor(objective='quantile', alpha=q, n_estimators=300,
                                   learning_rate=0.03, max_depth=3, min_child_samples=50,
                                   subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
        model.fit(X_train, y_train)
        models[f'{horizon}_q{int(q*100)}'] = model
        print(f"Trained: {horizon} - quantile {q}")

print(models.keys())

Trained: day1 - quantile 0.1
Trained: day1 - quantile 0.5
Trained: day1 - quantile 0.9
Trained: day2 - quantile 0.1
Trained: day2 - quantile 0.5
Trained: day2 - quantile 0.9
Trained: day3 - quantile 0.1
Trained: day3 - quantile 0.5
Trained: day3 - quantile 0.9
dict_keys(['day1_q10', 'day1_q50', 'day1_q90', 'day2_q10', 'day2_q50', 'day2_q90', 'day3_q10', 'day3_q50', 'day3_q90'])


In [ ]:
results = []

for horizon, (y_train, y_test) in targets.items():
    preds_pct = {}
    for q in [0.1, 0.5, 0.9]:   # Changed from [0.05, 0.5, 0.95]
        model = models[f'{horizon}_q{int(q*100)}']
        preds_pct[q] = model.predict(X_test)

    current_prices = test['current_gold_price'].values
    preds_price = {q: current_prices * (1 + preds_pct[q]) for q in [0.1, 0.5, 0.9]}  # Changed from [0.05, 0.5, 0.95]

    actual_price = current_prices * (1 + y_test.values)

    within_range = np.mean((actual_price >= preds_price[0.1]) & (actual_price <= preds_price[0.9]))  # Changed from preds_price[0.05] and preds_price[0.95]
    mae = np.mean(np.abs(actual_price - preds_price[0.5]))

    print(f"{horizon}: {within_range*100:.1f}% coverage | Median MAE: ${mae:.2f}")

    for q in [0.1, 0.5, 0.9]:   # Changed from [0.05, 0.5, 0.95]
        loss = pinball_loss(y_test.values, preds_pct[q], q)
        results.append({'horizon': horizon, 'quantile': q, 'pinball_loss': round(loss, 5)})

results_df = pd.DataFrame(results)
print(results_df.to_markdown())

day1: 63.8% coverage | Median MAE: $62.10
day2: 64.9% coverage | Median MAE: $87.34
day3: 62.2% coverage | Median MAE: $112.99
|    | horizon   |   quantile |   pinball_loss |
|---:|:----------|-----------:|---------------:|
|  0 | day1      |        0.1 |        0.00423 |
|  1 | day1      |        0.5 |        0.00678 |
|  2 | day1      |        0.9 |        0.00334 |
|  3 | day2      |        0.1 |        0.0061  |
|  4 | day2      |        0.5 |        0.00953 |
|  5 | day2      |        0.9 |        0.00417 |
|  6 | day3      |        0.1 |        0.0074  |
|  7 | day3      |        0.5 |        0.01231 |
|  8 | day3      |        0.9 |        0.00488 |


In [ ]:
import pickle

with open('gold_models.pkl', 'wb') as f:
    pickle.dump(models, f)

with open('feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("Models saved!")

Models saved!


In [ ]:
from google.colab import files
files.download('gold_models.pkl')
files.download('feature_cols.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>